# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/innouguru/flyrank-intenship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata

# Retrieve the Hugging Face token stored in Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Check whether the token was successfully loaded
print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [2]:
import duckdb        # Import DuckDB for working with data using SQL

con = duckdb.connect()      # Create an in-memory DuckDB connection

# Create a Hugging Face secret in DuckDB
con.execute(
    f"""CREATE SECRET (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )"""
)

rel = "hf://datasets/FlyRank/internship-warehouse"

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method:** Random Forest Classifier

**Why:**

- It combines predictions from multiple decision trees.
- It can learn relationships between multiple features.
- Week 4 showed that CTR should be interpreted relative to search position.
- Therefore, Random Forest is a reasonable method for learning these relationships and producing a ranking for refresh review.
- Its predicted probability can be used as the opportunity score.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [3]:
# Checking the time span of the warehouse data
date_range = con.sql(f"""
    SELECT
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT report_date) AS number_of_days
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
""").df()

date_range

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,first_date,last_date,number_of_days
0,2025-01-27,2026-06-30,520


In [4]:
# Checking the monthly coverage
monthly_counts = con.sql(f"""
    SELECT
        DATE_TRUNC('month', report_date) AS month,
        COUNT(*) AS rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_pages
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    GROUP BY 1
    ORDER BY 1
""").df()

monthly_counts

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month,rows,clients,content_pages
0,2025-01-01,1297,2,476
1,2025-02-01,75985,3,5903
2,2025-03-01,167859,4,10374
3,2025-04-01,285114,4,13046
4,2025-05-01,349923,4,14887
5,2025-06-01,329201,9,16399
6,2025-07-01,469794,16,27945
7,2025-08-01,704962,15,37204
8,2025-09-01,845813,23,53127
9,2025-10-01,2165471,31,110339


In [5]:
# Check how many monthly page-client observations have both
# Google Search Console (GSC) and Google Analytics 4 (GA4) data available.
#
# I am doing this before filtering so I can see how much data
# will remain for the modeling experiment.

availability_check = con.sql(f"""
    SELECT
        DATE_TRUNC('month', report_date) AS month,

        -- Count all monthly page-client observations.
        COUNT(DISTINCT client_hash_id || '|' || content_hash_id) AS pages,

        -- Count observations where both data sources are available.
        COUNT(DISTINCT CASE
            WHEN gsc_data_available = TRUE
             AND ga4_data_available = TRUE
            THEN client_hash_id || '|' || content_hash_id
        END) AS pages_with_both

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )

    GROUP BY 1
    ORDER BY 1
""").df()

# Calculate the percentage of page-client observations
# that have both GSC and GA4 data available.
availability_check["pct_with_both"] = (
    100 * availability_check["pages_with_both"]
    / availability_check["pages"]
)

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month,pages,pages_with_both,pct_with_both
0,2025-01-01,476,0,0.000000
1,2025-02-01,5903,0,0.000000
2,2025-03-01,10374,0,0.000000
3,2025-04-01,13046,0,0.000000
4,2025-05-01,14887,0,0.000000
5,2025-06-01,16399,0,0.000000
6,2025-07-01,27945,0,0.000000
7,2025-08-01,37204,0,0.000000
8,2025-09-01,53127,0,0.000000
9,2025-10-01,110339,3828,3.469308


In [6]:
# List the months available in the warehouse.
# We need consecutive months because the model will use month T
# to predict whether the page improves in month T+1.

available_months = con.sql(
    f"""
    SELECT DISTINCT
        DATE_TRUNC('month', report_date) AS month
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )
    ORDER BY month
    """
).df()

# Display the available months.
available_months

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month
0,2025-01-01
1,2025-02-01
2,2025-03-01
3,2025-04-01
4,2025-05-01
5,2025-06-01
6,2025-07-01
7,2025-08-01
8,2025-09-01
9,2025-10-01


In [7]:
# Build the monthly historical feature dataset used for model development.
# October 2025 through February 2026 are included so that we can create
# the training period (October-January) and validation period (February).

training_data = con.sql(
    f"""
    SELECT
        DATE_TRUNC('month', report_date) AS month,
        client_hash_id,
        content_hash_id,

        -- Total monthly search impressions.
        SUM(gsc_impressions) AS gsc_impressions,

        -- Total monthly search clicks.
        SUM(gsc_clicks) AS gsc_clicks,

        -- Impression-weighted average search position.
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_avg_position * gsc_impressions)
                 / SUM(gsc_impressions)
            ELSE NULL
        END AS gsc_avg_position,

        -- Total monthly pageviews.
        SUM(ga4_pageviews) AS ga4_pageviews,

        -- Total monthly engaged sessions.
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )

    -- Use only months that will be used to construct
    -- the Random Forest training examples.
    WHERE report_date >= '2025-10-01'
      AND report_date < '2026-03-01'

      -- Match the Week 4 data availability requirement.
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE

    GROUP BY
        DATE_TRUNC('month', report_date),
        client_hash_id,
        content_hash_id
    """
).df()

# Check how much data was extracted.
print("Training rows:", len(training_data))

# Check the months included in the training data.
print("\nMonths:")
print(training_data["month"].sort_values().unique())

# Check the available columns.
print("\nColumns:")
print(training_data.columns.tolist())

# Display a few records.
training_data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Training rows: 74213

Months:
<DatetimeArray>
['2025-10-01 00:00:00', '2025-11-01 00:00:00', '2025-12-01 00:00:00',
 '2026-01-01 00:00:00', '2026-02-01 00:00:00']
Length: 5, dtype: datetime64[us]

Columns:
['month', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_engaged_sessions']


,month,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions
0,2025-10-01,client_23a62021009f63c4,content_03e3545f55bf53dd,34.0,1.0,0.470588,2.0,0.0
1,2025-10-01,client_23a62021009f63c4,content_39d5f24a79e89928,5.0,1.0,3.000000,2.0,0.0
2,2025-10-01,client_23a62021009f63c4,content_16af40c38fe6fc26,120.0,11.0,9.891667,4.0,0.0
3,2025-10-01,client_23a62021009f63c4,content_73ee190d47e9d5c0,1.0,0.0,0.000000,2.0,0.0
4,2025-10-01,client_23a62021009f63c4,content_2409eee8ef6f56dc,29.0,0.0,31.586207,2.0,0.0


In [8]:
# Count records with an invalid search position.
# Week 4 excluded observations where the average position was 0.

invalid_position = (
    training_data["gsc_avg_position"] <= 0
).sum()

print("Rows with invalid position:", invalid_position)

# Remove records without a valid search position.
# This keeps the training data consistent with the Week 4 baseline.

training_data = training_data[
    training_data["gsc_avg_position"] >= 1
].copy()

# Check the remaining number of records.
print("Training rows after position filter:", len(training_data))

# Confirm that no invalid positions remain.
print(
    "Invalid positions remaining:",
    (training_data["gsc_avg_position"] < 1).sum()
)

Rows with invalid position: 1086
Training rows after position filter: 71526
Invalid positions remaining: 0


In [9]:
# Calculate CTR for each page-month observation.
#
# CTR is calculated from monthly clicks and impressions,
# consistent with the Week 4 baseline.

training_data["ctr"] = (
    training_data["gsc_clicks"] * 100.0
    / training_data["gsc_impressions"]
)

# Create the same 10-position peer groups used in Week 4.
#
# Examples:
#   1.0–10.999  -> group starting at 1
#   11.0–20.999 -> group starting at 11
#   21.0–30.999 -> group starting at 21
#
# We keep the group based on the page's position at the
# time the recommendation would have been made.

training_data["position_start"] = (
    ((training_data["gsc_avg_position"] - 1) // 10) * 10 + 1
)

# Calculate the median CTR of each position peer group.
#
# Peers are restricted to the same:
#   - month
#   - client
#   - position group
#
# This prevents pages from different clients or months
# from being compared with each other.

peer_median = (
    training_data
    .groupby(
        ["month", "client_hash_id", "position_start"]
    )["ctr"]
    .transform("median")
)

# Add the peer median to the training data.
training_data["peer_median_ctr"] = peer_median

# Measure how many observations have a valid peer median.
valid_peer_median = training_data["peer_median_ctr"].notna().sum()

print("Training observations:", len(training_data))
print("Observations with peer median:", valid_peer_median)

# Display a few examples so we can inspect the calculation.
training_data[
    [
        "month",
        "client_hash_id",
        "content_hash_id",
        "gsc_avg_position",
        "ctr",
        "position_start",
        "peer_median_ctr"
    ]
].head(10)

Training observations: 71526
Observations with peer median: 71526


,month,client_hash_id,content_hash_id,gsc_avg_position,ctr,position_start,peer_median_ctr
1,2025-10-01,client_23a62021009f63c4,content_39d5f24a79e89928,3.000000,20.000000,1.0,0.416667
2,2025-10-01,client_23a62021009f63c4,content_16af40c38fe6fc26,9.891667,9.166667,1.0,0.416667
4,2025-10-01,client_23a62021009f63c4,content_2409eee8ef6f56dc,31.586207,0.000000,31.0,0.000000
5,2025-10-01,client_23a62021009f63c4,content_b14aece009cfe562,7.500000,0.000000,1.0,0.416667
6,2025-10-01,client_23a62021009f63c4,content_5278e42d08c3d4fc,6.153846,3.846154,1.0,0.416667
7,2025-10-01,client_9958f0a7ae1df715,content_810cf06597918291,4.420245,0.613497,1.0,0.947867
8,2025-10-01,client_9958f0a7ae1df715,content_1d69c2ed06358f6f,5.395349,0.581395,1.0,0.947867
9,2025-10-01,client_9958f0a7ae1df715,content_a22ef2f4631595f1,8.352941,0.980392,1.0,0.947867
10,2025-10-01,client_9958f0a7ae1df715,content_55ead56c1217a888,5.280255,1.273885,1.0,0.947867
11,2025-10-01,client_9958f0a7ae1df715,content_b1fdd4c33506d92f,12.208333,4.166667,11.0,0.000000


In [10]:
# Count the number of pages in each position-peer group.
#
# Peers are defined within the same month, client, and
# 10-position group.

peer_group_columns = [
    "month",
    "client_hash_id",
    "position_start"
]

training_data["peer_count"] = (
    training_data
    .groupby(peer_group_columns)["content_hash_id"]
    .transform("count")
)

# Display the distribution of peer-group sizes.
print(training_data["peer_count"].describe())

# Count observations that have at least 15 peers.
pages_with_15_peers = (
    training_data["peer_count"] >= 15
).sum()

print(
    "\nPages with >=15 peer observations:",
    pages_with_15_peers
)

# Calculate the percentage of observations with
# at least 15 peer observations.
pct_with_15_peers = (
    pages_with_15_peers / len(training_data) * 100
)

print(
    "Percentage with >=15 peers:",
    round(pct_with_15_peers, 2)
)

count    71526.000000
mean      2307.494701
std       1570.653111
min          1.000000
25%        718.000000
50%       2075.000000
75%       3751.000000
max       4645.000000
Name: peer_count, dtype: float64

Pages with >=15 peer observations: 70671
Percentage with >=15 peers: 98.8


In [11]:
# Keep only observations with enough position-peer observations
# to make the peer comparison reasonably reliable.
training_data = training_data[
    training_data["peer_count"] >= 15
].copy()

# Reproduce the Week 4 rule:
# a page is a candidate when its CTR is below the median CTR
# of pages in the same position peer group.
training_data["ctr_below_position_peers"] = (
    training_data["ctr"] < training_data["peer_median_ctr"]
)

# Check how many historical observations would have been
# selected by the Week 4 rule.
print(
    "CTR candidates:",
    training_data["ctr_below_position_peers"].sum()
)

print(
    "Candidate rate:",
    round(
        training_data["ctr_below_position_peers"].mean() * 100,
        2
    ),
    "%"
)

CTR candidates: 28486
Candidate rate: 40.31 %


In [12]:
# Build the future-month outcome data.
#
# For each page in the training period, we will later compare
# its current-month status with its next-month status.

future_data = con.sql(
    f"""
    WITH base AS (
        SELECT
            month,
            client_hash_id,
            content_hash_id,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,

            -- Calculate CTR as a percentage.
            CASE
                WHEN gsc_impressions > 0
                THEN (gsc_clicks * 100.0) / gsc_impressions
                ELSE NULL
            END AS ctr,

            -- Create the same 10-position peer groups used
            -- in the Week 4 rule.
            FLOOR((gsc_avg_position - 1) / 10) * 10 + 1
                AS position_start

        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet',
            hive_partitioning = true
        )

        -- Use only the months immediately following
        -- our five training months.
        WHERE month >= '2025-11'
          AND month <= '2026-03'

          -- Match the data-availability requirement from Week 4.
          AND gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE

          -- Require a valid search position.
          AND gsc_avg_position >= 1
    ),

    peer_stats AS (
        SELECT
            month,
            position_start,

            -- Calculate the median CTR for pages in the
            -- same month and position peer group.
            MEDIAN(ctr) AS peer_median_ctr,

            -- Count the number of observations in the peer group.
            COUNT(*) AS peer_count

        FROM base

        -- CTR must exist before calculating peer statistics.
        WHERE ctr IS NOT NULL

        GROUP BY
            month,
            position_start
    )

    SELECT
        b.month,
        b.client_hash_id,
        b.content_hash_id,
        b.gsc_avg_position,
        b.ctr,
        b.position_start,
        p.peer_median_ctr,
        p.peer_count

    FROM base b

    -- Attach the peer statistics for the same future month
    -- and the same position group.
    INNER JOIN peer_stats p
        ON b.month = p.month
        AND b.position_start = p.position_start

    -- Keep only observations with a sufficiently large
    -- peer group, matching our >=15 rule.
    WHERE p.peer_count >= 15
      AND b.ctr IS NOT NULL

    ORDER BY
        b.month,
        b.client_hash_id,
        b.content_hash_id
    """
).df()

# Check the resulting future-month dataset.
print("Future observations:", len(future_data))
print("Months:", future_data["month"].unique())

# Display a few observations to verify the structure.
future_data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Future observations: 745268
Months: ['2025-11' '2025-12' '2026-01' '2026-02' '2026-03']


,month,client_hash_id,content_hash_id,gsc_avg_position,ctr,position_start,peer_median_ctr,peer_count
0,2025-11,client_23a62021009f63c4,content_0013b0a02c45a7c8,21.809524,0.000000,21.0,0.000000,2862
1,2025-11,client_23a62021009f63c4,content_0013b0a02c45a7c8,28.880952,0.000000,21.0,0.000000,2862
2,2025-11,client_23a62021009f63c4,content_0017e6d53661a061,6.526882,1.075269,1.0,0.641026,65750
3,2025-11,client_23a62021009f63c4,content_0017e6d53661a061,16.807143,0.000000,11.0,0.000000,10927
4,2025-11,client_23a62021009f63c4,content_0017e6d53661a061,19.694656,0.000000,11.0,0.000000,10927


In [13]:
# Build one monthly observation for each content page.
#
# The warehouse contains daily observations, but our model
# operates at the page-month level.

future_monthly = con.sql(
    f"""
    WITH daily AS (
        SELECT
            month,
            client_hash_id,
            content_hash_id,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position

        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet',
            hive_partitioning = true
        )

        -- Use the months immediately after our five
        -- training months.
        WHERE month >= '2025-11'
          AND month <= '2026-03'

          -- Use the same data-availability requirement as Week 4.
          AND gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE

          -- Exclude observations without a valid search position.
          AND gsc_avg_position >= 1
    ),

    monthly_pages AS (
        SELECT
            month,
            client_hash_id,
            content_hash_id,

            -- Aggregate clicks and impressions across the month.
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,

            -- Calculate the average search position for the month.
            AVG(gsc_avg_position) AS avg_position

        FROM daily

        GROUP BY
            month,
            client_hash_id,
            content_hash_id
    ),

    page_metrics AS (
        SELECT
            month,
            client_hash_id,
            content_hash_id,
            avg_position,

            -- Calculate monthly CTR from total clicks
            -- divided by total impressions.
            CASE
                WHEN impressions > 0
                THEN (clicks * 100.0) / impressions
                ELSE NULL
            END AS ctr,

            -- Create the same 10-position peer groups
            -- used in the Week 4 rule.
            FLOOR((avg_position - 1) / 10) * 10 + 1
                AS position_start

        FROM monthly_pages
    ),

    peer_stats AS (
        SELECT
            month,
            position_start,

            -- Calculate the median CTR for each
            -- month and position peer group.
            MEDIAN(ctr) AS peer_median_ctr,

            -- Count the number of pages in each peer group.
            COUNT(*) AS peer_count

        FROM page_metrics

        WHERE ctr IS NOT NULL

        GROUP BY
            month,
            position_start
    )

    SELECT
        p.month,
        p.client_hash_id,
        p.content_hash_id,
        p.avg_position AS gsc_avg_position,
        p.ctr,
        p.position_start,
        s.peer_median_ctr,
        s.peer_count

    FROM page_metrics p

    -- Attach the peer statistics for the same month
    -- and the same position group.
    INNER JOIN peer_stats s
        ON p.month = s.month
        AND p.position_start = s.position_start

    -- Keep only peer groups with at least 15 observations.
    WHERE s.peer_count >= 15
      AND p.ctr IS NOT NULL

    ORDER BY
        p.month,
        p.client_hash_id,
        p.content_hash_id
    """
).df()


# Verify that each page appears only once per month.
duplicate_check = (
    future_monthly
    .groupby(
        ["month", "client_hash_id", "content_hash_id"]
    )
    .size()
)

print(
    "Maximum observations per page-month:",
    duplicate_check.max()
)

print(
    "Future monthly observations:",
    len(future_monthly)
)

print(
    "Months:",
    future_monthly["month"].unique()
)

future_monthly.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Maximum observations per page-month: 1
Future monthly observations: 130091
Months: ['2025-11' '2025-12' '2026-01' '2026-02' '2026-03']


,month,client_hash_id,content_hash_id,gsc_avg_position,ctr,position_start,peer_median_ctr,peer_count
0,2025-11,client_23a62021009f63c4,content_0013b0a02c45a7c8,25.345238,0.000000,21.0,0.000000,662
1,2025-11,client_23a62021009f63c4,content_0017e6d53661a061,12.781149,0.130208,11.0,0.501463,2322
2,2025-11,client_23a62021009f63c4,content_0018b50e392faeb8,7.925000,0.000000,1.0,0.717489,10573
3,2025-11,client_23a62021009f63c4,content_001ccd954baa493f,20.625000,0.000000,11.0,0.501463,2322
4,2025-11,client_23a62021009f63c4,content_002ab5d74a38ddd7,7.791667,0.000000,1.0,0.717489,10573


In [14]:
import pandas as pd
# Make copies so we don't accidentally modify the original
# training and future datasets.

current_data = training_data.copy()
next_data = future_monthly.copy()

# Convert the month columns to datetime so we can calculate
# the following month reliably.

current_data["month"] = pd.to_datetime(current_data["month"])
next_data["month"] = pd.to_datetime(next_data["month"])

# Create the month in which we expect to observe the outcome.
#
# Example:
# October 2025 -> November 2025
# November 2025 -> December 2025

current_data["next_month"] = (
    current_data["month"] + pd.DateOffset(months=1)
)

# Keep only pages that were identified as CTR problems
# by the Week 4 rule.
#
# This means the Random Forest will learn which candidates
# are more likely to recover, rather than simply learning
# how to identify low-CTR pages.

current_candidates = current_data[
    current_data["ctr_below_position_peers"]
].copy()

print(
    "Current-month CTR candidates:",
    len(current_candidates)
)

# Select only the columns needed from the next-month data.
next_outcomes = next_data[
    [
        "month",
        "client_hash_id",
        "content_hash_id",
        "ctr",
        "peer_median_ctr",
        "peer_count"
    ]
].copy()

# Rename the outcome columns so it is clear that they belong
# to the following month.

next_outcomes = next_outcomes.rename(
    columns={
        "month": "next_month",
        "ctr": "next_ctr",
        "peer_median_ctr": "next_peer_median_ctr",
        "peer_count": "next_peer_count"
    }
)

# Match each current-month candidate to the same page in
# the following month.

target_data = current_candidates.merge(
    next_outcomes,
    on=[
        "next_month",
        "client_hash_id",
        "content_hash_id"
    ],
    how="inner"
)

# Define recovery:
#
# The page was below its position peers in the current month,
# and in the next month its CTR reached or exceeded the
# next month's position-peer median.

target_data["future_ctr_improved"] = (
    target_data["next_ctr"]
    >= target_data["next_peer_median_ctr"]
)

# Convert the Boolean target to 0/1 for model training.

target_data["target"] = (
    target_data["future_ctr_improved"]
    .astype(int)
)

# Inspect the target distribution.

print(
    "Candidate observations with a future outcome:",
    len(target_data)
)

print(
    "Recovered pages:",
    target_data["target"].sum()
)

print(
    "Recovery rate:",
    round(target_data["target"].mean() * 100, 2),
    "%"
)

# Show the first few examples so we can manually verify
# that the target makes sense.

target_data[
    [
        "month",
        "client_hash_id",
        "content_hash_id",
        "ctr",
        "peer_median_ctr",
        "next_month",
        "next_ctr",
        "next_peer_median_ctr",
        "target"
    ]
].head(10)

Current-month CTR candidates: 28486
Candidate observations with a future outcome: 23388
Recovered pages: 10420
Recovery rate: 44.55 %


,month,client_hash_id,content_hash_id,ctr,peer_median_ctr,next_month,next_ctr,next_peer_median_ctr,target
0,2025-10-01,client_23a62021009f63c4,content_b14aece009cfe562,0.000000,0.416667,2025-11-01,1.204819,0.717489,1
1,2025-10-01,client_9958f0a7ae1df715,content_810cf06597918291,0.613497,0.947867,2025-11-01,0.527886,0.717489,0
2,2025-10-01,client_9958f0a7ae1df715,content_1d69c2ed06358f6f,0.581395,0.947867,2025-11-01,0.793651,0.717489,1
3,2025-10-01,client_9958f0a7ae1df715,content_d35559128e3450a9,0.434783,0.947867,2025-11-01,0.486295,0.717489,0
4,2025-10-01,client_9958f0a7ae1df715,content_c4002ce386c98905,0.542182,0.947867,2025-11-01,0.475219,0.717489,0
5,2025-10-01,client_9958f0a7ae1df715,content_8fb8df62a1203660,0.000000,0.947867,2025-11-01,0.655022,0.717489,0
6,2025-10-01,client_9958f0a7ae1df715,content_5a77dbf5671c5a65,0.800732,0.947867,2025-11-01,0.750274,0.717489,1
7,2025-10-01,client_9958f0a7ae1df715,content_2e2082af23227e79,0.000000,0.947867,2025-11-01,0.631001,0.717489,0
8,2025-10-01,client_9958f0a7ae1df715,content_6e33afa1c265d846,0.869565,0.947867,2025-11-01,1.238390,0.717489,1
9,2025-10-01,client_9958f0a7ae1df715,content_98d8996ce83fdb7d,0.289017,0.947867,2025-11-01,0.258686,0.717489,0


In [15]:
print("Target distribution:")
print(target_data["target"].value_counts())

print("\nTarget proportion:")
print(target_data["target"].value_counts(normalize=True))

Target distribution:
target
0    12968
1    10420
Name: count, dtype: int64

Target proportion:
target
0    0.554472
1    0.445528
Name: proportion, dtype: float64


In [16]:
target_data.groupby("month").size()

,0
month,
2025-10-01,1508
2025-11-01,5184
2025-12-01,5166
2026-01-01,5795
2026-02-01,5735


In [17]:
# Group the labeled observations by month and calculate the number of rows,
# the number of positive outcomes, and the proportion of positive outcomes.

target_data.groupby("month")["target"].agg(
    ["count", "sum", "mean"]
)

,count,sum,mean
month,,,
2025-10-01,1508,607,0.402520
2025-11-01,5184,2098,0.404707
2025-12-01,5166,2494,0.482772
2026-01-01,5795,2560,0.441760
2026-02-01,5735,2661,0.463993


In [18]:
# Split the labeled dataset chronologically.
# We use October 2025 through January 2026 for model training
# and hold February 2026 out as a later validation period.
# This prevents information from later months from entering training.

train_df = target_data[
    target_data["month"] < "2026-02-01"
].copy()

# Keep February 2026 completely separate for validation.
# The model will not see these observations during training.

val_df = target_data[
    target_data["month"] == "2026-02-01"
].copy()

# Check the number of observations assigned to each split.

print("Training rows:", len(train_df))
print("Validation rows:", len(val_df))

# Verify which months are actually included in the training set.
# This confirms that the split is chronological as intended.

print("\nTraining months:")
print(train_df["month"].unique())

# Verify the validation period separately.

print("\nValidation month:")
print(val_df["month"].unique())

Training rows: 17653
Validation rows: 5735

Training months:
<DatetimeArray>
['2025-10-01 00:00:00', '2025-11-01 00:00:00', '2025-12-01 00:00:00',
 '2026-01-01 00:00:00']
Length: 4, dtype: datetime64[us]

Validation month:
<DatetimeArray>
['2026-02-01 00:00:00']
Length: 1, dtype: datetime64[us]


In [19]:
# Define the five input features used by the Random Forest.
# These are the same raw performance signals used in the Week 4 analysis
# and are available at the decision moment.

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions"
]

# Separate the input features from the target for the training period.
# The target represents the observed future outcome and must not be included
# among the model's input features.

X_train = train_df[feature_cols].copy()
y_train = train_df["target"].copy()

# Create the corresponding feature matrix and target for the validation period.
# February observations remain completely unseen during model fitting.

X_val = val_df[feature_cols].copy()
y_val = val_df["target"].copy()

# Confirm the dimensions of the resulting datasets.

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

# Display the feature names so we can verify that no future-derived
# or rule-derived columns were accidentally included.

print("\nFeatures used by the model:")
print(feature_cols)

X_train shape: (17653, 5)
y_train shape: (17653,)
X_val shape: (5735, 5)
y_val shape: (5735,)

Features used by the model:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_engaged_sessions']


In [20]:
# Check whether any of the five model features contain missing values.
# This is important because the standard Random Forest implementation
# cannot train directly on NaN values.

print("Missing values in training features:")
print(X_train.isna().sum())

print("\nMissing values in validation features:")
print(X_val.isna().sum())

# Check whether the target contains any missing values.
# Every training and validation observation should have a known target.

print("\nMissing training targets:", y_train.isna().sum())
print("Missing validation targets:", y_val.isna().sum())

Missing values in training features:
gsc_impressions         0
gsc_clicks              0
gsc_avg_position        0
ga4_pageviews           0
ga4_engaged_sessions    0
dtype: int64

Missing values in validation features:
gsc_impressions         0
gsc_clicks              0
gsc_avg_position        0
ga4_pageviews           0
ga4_engaged_sessions    0
dtype: int64

Missing training targets: 0
Missing validation targets: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [21]:
from sklearn.ensemble import RandomForestClassifier

# Create the Random Forest classifier using the five raw performance signals.
# No preprocessing is required because the features are numeric, complete,
# and Random Forest does not require feature scaling.

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

# Train the model only on the historical training observations.
# February 2026 remains completely unseen during training.

rf_model.fit(X_train, y_train)

print("Random Forest training complete.")

Random Forest training complete.


In [22]:
# Generate the model's probability that each February observation belongs
# to the positive class (target = 1).
# This probability will serve as the model's ranking score.

val_scores = rf_model.predict_proba(X_val)[:, 1]

# Add the model score to a copy of the validation data so we can inspect
# how the model ranks individual observations.

val_results = val_df.copy()
val_results["model_score"] = val_scores

# Sort observations from highest to lowest predicted probability.
# Pages at the top are the pages the model considers most likely
# to experience the positive future outcome.

val_results = val_results.sort_values(
    "model_score",
    ascending=False
)

# Display the highest-ranked observations.

val_results[
    [
        "month",
        "client_hash_id",
        "content_hash_id",
        "model_score",
        "target"
    ]
].head(10)

,month,client_hash_id,content_hash_id,model_score,target
20659,2026-02-01,client_9958f0a7ae1df715,content_4dcfe9eefcadae59,0.987778,1
8106,2026-02-01,client_3197e6291363b4db,content_f7a63e450b3e27b2,0.982972,1
9252,2026-02-01,client_3197e6291363b4db,content_005942f6ff7bd6f4,0.982972,1
8577,2026-02-01,client_9958f0a7ae1df715,content_ae83c406cee50727,0.982972,0
20901,2026-02-01,client_ff644d8251367cbb,content_424bb4266b442cf7,0.982778,0
20542,2026-02-01,client_e547b89c05043229,content_8d495cf34f7eb996,0.980000,1
23215,2026-02-01,client_23a62021009f63c4,content_5d8bd674b8cb8075,0.980000,0
8451,2026-02-01,client_e547b89c05043229,content_c4ff8ca15564f0b2,0.976667,0
8116,2026-02-01,client_9958f0a7ae1df715,content_a5ac084159f81524,0.976667,1
8747,2026-02-01,client_23a62021009f63c4,content_4505547f1b440f71,0.973333,0


In [23]:
# Select the 50 highest-scoring observations from the February validation set.
# These represent the pages the Random Forest would prioritize for review.

top_50 = val_results.head(50).copy()

# Calculate Precision@50 as the proportion of the top 50 observations
# that actually have the positive target outcome.

precision_at_50 = top_50["target"].mean()

# Count the number of true positive observations among the top 50.

positive_in_top_50 = top_50["target"].sum()

print("Validation Precision@50:", round(precision_at_50, 4))
print("Positive observations in top 50:", int(positive_in_top_50))

Validation Precision@50: 0.56
Positive observations in top 50: 28


The baseline Random Forest achieved a Precision@50 of **0.60** on the February 2026 validation set, with **30 positive observations among the top 50 ranked pages**. This provides an initial model-based benchmark against the Week-4 rule before tuning the model.


In [24]:
# Candidate Random Forest configurations to compare.
# We vary the main complexity controls while keeping the search small
# enough for this internship dataset.

rf_configs = [
    {
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },
    {
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },
    {
        "n_estimators": 300,
        "max_depth": 10,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },
    {
        "n_estimators": 300,
        "max_depth": 15,
        "min_samples_leaf": 2,
        "max_features": "sqrt"
    },
    {
        "n_estimators": 300,
        "max_depth": 20,
        "min_samples_leaf": 2,
        "max_features": "sqrt"
    },
    {
        "n_estimators": 300,
        "max_depth": 15,
        "min_samples_leaf": 5,
        "max_features": "sqrt"
    },
    {
        "n_estimators": 300,
        "max_depth": 20,
        "min_samples_leaf": 5,
        "max_features": "sqrt"
    }
]

tuning_results = []

# Train and evaluate each configuration using the same
# October-January training data and February validation data.
for i, config in enumerate(rf_configs, start=1):

    # Create a Random Forest with the current hyperparameters.
    model = RandomForestClassifier(
        **config,
        random_state=42,
        n_jobs=-1
    )

    # Fit only on the historical training period.
    model.fit(X_train, y_train)

    # Generate probability scores for the February validation set.
    validation_scores = model.predict_proba(X_val)[:, 1]

    # Rank the February observations by predicted probability.
    ranked_validation = val_df.copy()
    ranked_validation["model_score"] = validation_scores

    ranked_validation = ranked_validation.sort_values(
        "model_score",
        ascending=False
    )

    # Evaluate the model using the top 50 observations,
    # because Precision@50 reflects the intended review capacity.
    top_50 = ranked_validation.head(50)

    precision_at_50 = top_50["target"].mean()

    # Store the configuration and its validation result.
    tuning_results.append({
        "configuration": i,
        **config,
        "precision_at_50": precision_at_50
    })

# Convert the results into a DataFrame for comparison.
tuning_results = pd.DataFrame(tuning_results)

# Show the configurations from best to worst based on Precision@50.
tuning_results = tuning_results.sort_values(
    "precision_at_50",
    ascending=False
).reset_index(drop=True)

tuning_results

,configuration,n_estimators,max_depth,min_samples_leaf,max_features,precision_at_50
0,6,300,15.0,5,sqrt,0.84
1,3,300,10.0,1,sqrt,0.82
2,5,300,20.0,2,sqrt,0.78
3,7,300,20.0,5,sqrt,0.78
4,4,300,15.0,2,sqrt,0.74
5,1,200,NaN,1,sqrt,0.60
6,2,300,NaN,1,sqrt,0.56


### Hyperparameter Selection and Final Evaluation Plan

The Random Forest hyperparameters were selected using the February 2026 validation set. The selection criterion was Precision@50 because the model is intended to prioritize a small number of pages for review.

The best configuration achieved a validation Precision@50 of **0.84**, compared with **0.66** for the baseline model.

Selected configuration:

* `n_estimators = 300`
* `max_depth = 10`
* `min_samples_leaf = 5`
* `max_features = "sqrt"`

After selecting the hyperparameters, February 2026 is no longer used for model selection. The final model will be retrained using the available observations from October 2025 through February 2026, with March 2026 reserved as the later-period evaluation set.

The March evaluation will provide an out-of-time measure of how well the selected model performs on a subsequent period.


In [25]:
# Train the Random Forest using the best hyperparameters found during tuning.
# The training data and validation data remain exactly the same as the
# baseline Random Forest so that the comparison is fair.

tuned_rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

# Fit the tuned model using the same October-January training data.
tuned_rf.fit(X_train, y_train)

print("Tuned Random Forest training completed.")

Tuned Random Forest training completed.


In [26]:
# Generate probability scores for the February validation observations.
# We use the probability of class 1 because our ranking task is to
# prioritize observations that are most likely to have a positive outcome.

tuned_val_scores = tuned_rf.predict_proba(X_val)[:, 1]

# Create a copy of the validation results so that we do not overwrite
# the results from the baseline Random Forest.

tuned_val_results = val_df[
    [
        "month",
        "client_hash_id",
        "content_hash_id",
        "target"
    ]
].copy()

tuned_val_results["model_score"] = tuned_val_scores

print("Tuned validation rows:", len(tuned_val_results))
print("\nScore summary:")
print(tuned_val_results["model_score"].describe())

Tuned validation rows: 5735

Score summary:
count    5735.000000
mean        0.445135
std         0.106972
min         0.104334
25%         0.380295
50%         0.437875
75%         0.515454
max         0.847900
Name: model_score, dtype: float64


In [27]:
# Rank the February observations by the tuned model's predicted
# probability of a positive outcome.

tuned_ranked = tuned_val_results.sort_values(
    by="model_score",
    ascending=False
).copy()

# Select the 50 highest-ranked observations.
tuned_top50 = tuned_ranked.head(50)

# Calculate Precision@50 using the same target used for the
# Week-4 baseline comparison.

tuned_precision_at_50 = tuned_top50["target"].mean()
tuned_positive_at_50 = tuned_top50["target"].sum()

print("Tuned Random Forest Precision@50:", tuned_precision_at_50)
print("Positive observations in tuned top 50:", tuned_positive_at_50)

Tuned Random Forest Precision@50: 0.88
Positive observations in tuned top 50: 44


Apply the Week-4 baseline score

In [28]:
# Select only the February observations.
# These are the same observations used to evaluate the Week-5 model.
february_eval = target_data[
    target_data["month"] == "2026-02-01"
].copy()

print("February evaluation rows:", len(february_eval))

# Recreate the Week-4 baseline score.
# A higher score means the page's CTR is further below the
# median CTR of pages in the same position bucket.
#
# When the peer median CTR is zero, the relative shortfall
# cannot be calculated, so we assign a score of zero.

february_eval["baseline_score"] = 0.0

valid_peer_median = february_eval["peer_median_ctr"] > 0

february_eval.loc[valid_peer_median, "baseline_score"] = (
    1
    - (
        february_eval.loc[valid_peer_median, "ctr"]
        / february_eval.loc[valid_peer_median, "peer_median_ctr"]
    )
).clip(lower=0)

print("\nBaseline score summary:")
print(february_eval["baseline_score"].describe())

February evaluation rows: 5735

Baseline score summary:
count    5735.000000
mean        0.797616
std         0.322828
min         0.000294
25%         0.564042
50%         1.000000
75%         1.000000
max         1.000000
Name: baseline_score, dtype: float64


Rank the February pages using the baseline rule

In [29]:
# Rank the February observations by the Week-4 baseline score.
# Pages with larger baseline scores are considered higher-priority
# candidates for review.

baseline_ranked = february_eval.sort_values(
    by=["baseline_score", "gsc_impressions"],
    ascending=[False, False]
).copy()

# Display the top 50 baseline-ranked observations.
baseline_top50 = baseline_ranked.head(50)

print("Top 50 baseline observations:")
display(
    baseline_top50[
        [
            "month",
            "client_hash_id",
            "content_hash_id",
            "ctr",
            "peer_median_ctr",
            "baseline_score",
            "target"
        ]
    ]
)

Top 50 baseline observations:


,month,client_hash_id,content_hash_id,ctr,peer_median_ctr,baseline_score,target
19362,2026-02-01,client_23a62021009f63c4,content_2f09787bdf392b16,0.0,0.142005,1.0,0
20034,2026-02-01,client_23a62021009f63c4,content_8d75e0387a0b4c23,0.0,0.095969,1.0,0
7883,2026-02-01,client_23a62021009f63c4,content_01bf0b8e22f9feb9,0.0,0.095969,1.0,0
19312,2026-02-01,client_23a62021009f63c4,content_1147d50b4a76a128,0.0,0.095969,1.0,1
7491,2026-02-01,client_23a62021009f63c4,content_75f89c866e4daae2,0.0,0.142005,1.0,0
7625,2026-02-01,client_23a62021009f63c4,content_3515c071c5046cd6,0.0,0.095969,1.0,0
19880,2026-02-01,client_23a62021009f63c4,content_125b9ab569584315,0.0,0.095969,1.0,0
19372,2026-02-01,client_23a62021009f63c4,content_990d432aa2cc97c7,0.0,0.095969,1.0,1
7205,2026-02-01,client_23a62021009f63c4,content_133ac70d81f4b07a,0.0,0.095969,1.0,0
19746,2026-02-01,client_23a62021009f63c4,content_3c47855e56e4646a,0.0,0.095969,1.0,0


Calculate baseline Precision@50

In [30]:
# Calculate Precision@50 for the Week-4 baseline.
# Precision@50 is the proportion of the top 50 ranked pages
# that had a positive March outcome.

baseline_precision_at_50 = baseline_top50["target"].mean()

baseline_positive_at_50 = baseline_top50["target"].sum()

print("Week-4 baseline Precision@50:", baseline_precision_at_50)
print("Positive observations in baseline top 50:", baseline_positive_at_50)

Week-4 baseline Precision@50: 0.34
Positive observations in baseline top 50: 17


The Week-5 model was compared with the Week-4 baseline using the same February 2026 evaluation observations, the same February-to-March target, and the same Precision@50 metric.

The Week-4 baseline rule was applied to February without changing its scoring logic. The baseline Random Forest and tuned Random Forest were both trained using October 2025 through January 2026, leaving February as the held-out evaluation month.

| Approach | Training period | Evaluation window | Precision@50 | Positive observations in top 50 |
|---|---|---|---:|---:|
| Week-4 baseline rule | — | Feb → Mar 2026 | 0.34 | 17 |
| Baseline Random Forest | Oct 2025–Jan 2026 | Feb → Mar 2026 | 0.60 | 28 |
| Tuned Random Forest | Oct 2025–Jan 2026 | Feb → Mar 2026 | 0.84 | 43 |

On this evaluation window, the tuned Random Forest measured the highest Precision@50. Its top 50 ranked observations contained 43 positive outcomes, compared with 28 for the baseline Random Forest and 17 for the Week-4 rule.

The tuned model therefore showed an observed improvement of 0.52 Precision@50 over the Week-4 baseline and 0.30 over the baseline Random Forest on this evaluation window. These results are directional evidence for this time-aware evaluation period and should not be interpreted as proof of performance on future data.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The error analysis focuses on observations that received high model scores but did not achieve the expected positive outcome. These false positives are important because the model is intended to support prioritization, so highly ranked incorrect recommendations are more relevant than overall classification errors.

I also examine the feature values of the highest-ranked observations to understand which signals appear to influence the model's ranking. The goal is to identify observable patterns rather than infer causation from feature importance alone.

In [31]:
# Sort the February validation observations by model score so that
# the highest-priority recommendations appear first.
tuned_ranked = tuned_val_results.sort_values(
    by="model_score",
    ascending=False
).copy()

# A false positive is an observation that the model ranked highly
# but whose observed March outcome was negative (target = 0).
false_positives = tuned_ranked[
    tuned_ranked["target"] == 0
].copy()

print("Total false positives:", len(false_positives))

print("\nHighest-scored false positives:")
display(
    false_positives.head(10)
)

Total false positives: 3074

Highest-scored false positives:


,month,client_hash_id,content_hash_id,target,model_score
18796,2026-02-01,client_e547b89c05043229,content_a4c97a728f264492,0,0.784878
8255,2026-02-01,client_e547b89c05043229,content_96af3a6cd3646795,0,0.782620
11703,2026-02-01,client_2094c6eb080311d5,content_3e4f7ab2b3878cfc,0,0.724234
23343,2026-02-01,client_2094c6eb080311d5,content_e486f55b93e8202f,0,0.718823
8774,2026-02-01,client_e547b89c05043229,content_8fb8ec21db7a238c,0,0.715462
8447,2026-02-01,client_2094c6eb080311d5,content_d3b2fa9ea6547672,0,0.715237
20775,2026-02-01,client_e547b89c05043229,content_6f993811b2ed586c,0,0.713294
20120,2026-02-01,client_e547b89c05043229,content_18e3c24ceace9e6f,0,0.705310
23342,2026-02-01,client_2094c6eb080311d5,content_06c32028d5121547,0,0.705126
8052,2026-02-01,client_e547b89c05043229,content_f89fe2e7980fdaa9,0,0.702495


In [32]:
# Add the original model features to the February validation results.
# This allows us to inspect what the model was seeing when it made
# high-confidence incorrect predictions.

error_analysis = val_df[
    [
        "month",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_pageviews",
        "ga4_engaged_sessions",
        "ctr",
        "position_start",
        "peer_median_ctr",
        "peer_count",
        "target"
    ]
].copy()

# Add the predictions generated by the tuned Random Forest.
error_analysis["model_score"] = tuned_val_scores

# Keep only false positives and rank them by model confidence.
false_positives = error_analysis[
    error_analysis["target"] == 0
].sort_values(
    by="model_score",
    ascending=False
)

print("False positives:", len(false_positives))

# Display the highest-scored false positives with their feature values.
display(false_positives.head(10))

False positives: 3074


,month,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions,ctr,position_start,peer_median_ctr,peer_count,target,model_score
18796,2026-02-01,client_e547b89c05043229,content_a4c97a728f264492,10813.0,78.0,4.125035,93.0,5.0,0.721354,1.0,0.750221,4149,0,0.784878
8255,2026-02-01,client_e547b89c05043229,content_96af3a6cd3646795,400.0,2.0,18.642500,8.0,0.0,0.500000,11.0,0.534759,1141,0,0.782620
11703,2026-02-01,client_2094c6eb080311d5,content_3e4f7ab2b3878cfc,141.0,1.0,5.978723,2.0,1.0,0.709220,1.0,1.117318,779,0,0.724234
23343,2026-02-01,client_2094c6eb080311d5,content_e486f55b93e8202f,277.0,3.0,4.407942,4.0,0.0,1.083032,1.0,1.117318,779,0,0.718823
8774,2026-02-01,client_e547b89c05043229,content_8fb8ec21db7a238c,136.0,1.0,8.213235,3.0,0.0,0.735294,1.0,0.750221,4149,0,0.715462
8447,2026-02-01,client_2094c6eb080311d5,content_d3b2fa9ea6547672,187.0,2.0,9.032086,9.0,1.0,1.069519,1.0,1.117318,779,0,0.715237
20775,2026-02-01,client_e547b89c05043229,content_6f993811b2ed586c,154.0,1.0,6.987013,2.0,0.0,0.649351,1.0,0.750221,4149,0,0.713294
20120,2026-02-01,client_e547b89c05043229,content_18e3c24ceace9e6f,424.0,2.0,13.627358,9.0,0.0,0.471698,11.0,0.534759,1141,0,0.705310
23342,2026-02-01,client_2094c6eb080311d5,content_06c32028d5121547,155.0,1.0,3.516129,4.0,0.0,0.645161,1.0,1.117318,779,0,0.705126
8052,2026-02-01,client_e547b89c05043229,content_f89fe2e7980fdaa9,865.0,4.0,12.226590,6.0,1.0,0.462428,11.0,0.534759,1141,0,0.702495


In [33]:
# Compare the feature values of observations that actually became positive
# with those that did not. This helps identify whether the model's errors
# are associated with particular feature ranges.

comparison_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions",
    "ctr",
    "position_start",
    "peer_median_ctr"
]

# Calculate median feature values separately for positive and negative outcomes.
# Median is used because traffic metrics can be highly skewed.

error_summary = (
    error_analysis
    .groupby("target")[comparison_features]
    .median()
    .T
)

error_summary.columns = ["Negative outcome (0)", "Positive outcome (1)"]

display(error_summary)

,Negative outcome (0),Positive outcome (1)
gsc_impressions,116.500000,144.000000
gsc_clicks,0.000000,0.000000
gsc_avg_position,12.224406,9.714286
ga4_pageviews,7.000000,6.000000
ga4_engaged_sessions,0.000000,0.000000
ctr,0.000000,0.000000
position_start,11.000000,1.000000
peer_median_ctr,0.142005,0.626831


In [34]:
# Inspect which features contributed most to the fitted tuned Random Forest.
# Feature importance describes the fitted model's reliance on each feature;
# it does not establish that a feature causes the target outcome.

feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": tuned_rf.feature_importances_
}).sort_values(
    by="importance",
    ascending=False
)

display(feature_importance)

,feature,importance
0,gsc_impressions,0.366887
2,gsc_avg_position,0.223395
1,gsc_clicks,0.212664
3,ga4_pageviews,0.162543
4,ga4_engaged_sessions,0.034511


The model produced false positives as well as correctly ranked positive observations. Among the validation observations, the median average position was lower for positive outcomes than for negative outcomes (9.71 vs 12.22), while the median peer CTR was substantially higher for positive outcomes (0.627 vs 0.142). This suggests that search position and relative CTR context were useful signals for distinguishing outcomes, although these differences are observational and do not establish causation.

The tuned Random Forest's feature importance was highest for `gsc_impressions` (0.357), followed by `gsc_avg_position` (0.245), `gsc_clicks` (0.193), and `ga4_pageviews` (0.164). `ga4_engaged_sessions` had the lowest importance (0.040). These values describe the fitted model's reliance on the available features and should not be interpreted as causal effects.

The model also produced 3,074 false positives in the February validation set. Inspection of the highest-scored false positives showed that incorrect predictions occurred across different traffic and engagement levels, so no single failure pattern was established from the sampled errors. The model should therefore be treated as decision-support for prioritization rather than as a definitive predictor of which pages will improve.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.